# 아산시 데이터 탐색 (Data Check)
> 각 데이터 소스별 head, shape, 컬럼 타입, 기본 통계 확인


In [ ]:
import pandas as pd
import numpy as np
import zipfile, os, glob
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)

# 데이터 경로 (환경에 맞게 수정)
DATA_DIR = r"C:\Users\HP\Desktop\01.데이터"
print("데이터 경로:", DATA_DIR)
print("하위 폴더:", [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR,d))] if os.path.exists(DATA_DIR) else "경로 없음")

def read_zip_csv(zip_path, csv_name, nrows=5):
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open(csv_name) as f:
            for enc in ['utf-8-sig','utf-8','cp949','euc-kr']:
                try:
                    f.seek(0)
                    return pd.read_csv(f, nrows=nrows, encoding=enc)
                except: continue
    return None


---
## 1. 카드매출 데이터


In [ ]:
card_zips = sorted(glob.glob(os.path.join(DATA_DIR, "**/201901.zip"), recursive=True))
print(f"카드매출 ZIP: {len(card_zips)}개")
if card_zips:
    with zipfile.ZipFile(card_zips[0]) as zf:
        for n in sorted(zf.namelist()):
            print(f"  {n} ({zf.getinfo(n).file_size/1024:.0f}KB)")


In [ ]:
# 각 CSV head 확인
if card_zips:
    with zipfile.ZipFile(card_zips[0]) as zf:
        for csv_name in sorted(n for n in zf.namelist() if n.endswith('.csv')):
            df = read_zip_csv(card_zips[0], csv_name, nrows=3)
            if df is not None:
                print(f"\n{'='*80}")
                print(f"{csv_name} | 컬럼: {len(df.columns)}개")
                print(f"컬럼명: {list(df.columns)}")
                display(df.head(3))


### ASAN_CSTMR_DATA 상세


In [ ]:
if card_zips:
    with zipfile.ZipFile(card_zips[0]) as zf:
        f = [n for n in zf.namelist() if 'CSTMR' in n]
        if f:
            df = read_zip_csv(card_zips[0], f[0], nrows=5000)
            print(f"{f[0]} | Shape: {df.shape}")
            print(f"\n타입:\n{df.dtypes}")
            print(f"\nSEX: {df['SEX'].value_counts().to_dict()}")
            print(f"AGE: {sorted(df['AGE'].unique())}")
            print(f"\n통계:\n{df.describe()}")
            display(df.head())


---
## 2. KCB 신용정보


In [ ]:
kcb_zips = sorted(glob.glob(os.path.join(DATA_DIR, "**/201601.zip"), recursive=True))
print(f"KCB ZIP: {len(kcb_zips)}개")
if kcb_zips:
    with zipfile.ZipFile(kcb_zips[0]) as zf:
        for n in sorted(zf.namelist()):
            print(f"  {n} ({zf.getinfo(n).file_size/1024:.0f}KB) {'** PIA 암호화 **' if '.pia' in n else ''}")


In [ ]:
if kcb_zips:
    with zipfile.ZipFile(kcb_zips[0]) as zf:
        for csv_name in sorted(n for n in zf.namelist() if n.endswith('.csv')):
            df = read_zip_csv(kcb_zips[0], csv_name, nrows=5)
            if df is not None:
                print(f"\n{'='*80}")
                print(f"{csv_name} | 컬럼: {len(df.columns)}개")
                display(df.head(3))


---
## 3. T맵 내비게이션


In [ ]:
tmap_files = sorted(glob.glob(os.path.join(DATA_DIR, "**/as_tmap_od_*.csv"), recursive=True))
print(f"T맵 파일: {len(tmap_files)}개")
if tmap_files:
    df = pd.read_csv(tmap_files[0], nrows=1000, encoding='utf-8-sig')
    print(f"\n{os.path.basename(tmap_files[0])} | Shape: {df.shape}")
    print(f"\n타입:\n{df.dtypes}")
    print(f"\n목적지 카테고리 TOP 10:\n{df['dstn_ctgy'].value_counts().head(10)}")
    display(df.head())


---
## 4. SKT 유동인구 (대용량)


In [ ]:
skt_files = sorted(glob.glob(os.path.join(DATA_DIR, "**/AS_SKT_*"), recursive=True))
print(f"SKT 파일: {len(skt_files)}개")
for f in skt_files:
    sz = os.path.getsize(f)/(1024**3)
    print(f"  {os.path.basename(f)}: {sz:.1f}GB" if sz>0.1 else f"  {os.path.basename(f)}: {os.path.getsize(f)/1024/1024:.0f}MB")


In [ ]:
# SKT head만 (대용량 주의)
for f in skt_files[:4]:
    try:
        df = pd.read_csv(f, nrows=5, encoding='utf-8-sig')
    except:
        try: df = pd.read_csv(f, nrows=5, encoding='cp949')
        except: print(f"읽기 실패: {os.path.basename(f)}"); continue
    print(f"\n{'='*80}")
    print(f"{os.path.basename(f)} | 컬럼: {len(df.columns)}개")
    print(f"컬럼: {list(df.columns)}")
    display(df.head())


---
## 5. 인구 데이터


In [ ]:
pop_files = sorted(glob.glob(os.path.join(DATA_DIR, "**/AS_*POP*"), recursive=True))
pop_files += sorted(glob.glob(os.path.join(DATA_DIR, "**/*PPLTN*"), recursive=True))
pop_files += sorted(glob.glob(os.path.join(DATA_DIR, "**/*인구*"), recursive=True))
pop_files = list(set(pop_files))

print(f"인구 관련 파일: {len(pop_files)}개")
for f in pop_files[:5]:
    sz = os.path.getsize(f)/1024/1024
    print(f"  {os.path.basename(f)}: {sz:.0f}MB")

for f in pop_files[:3]:
    try: df = pd.read_csv(f, nrows=5, encoding='utf-8-sig')
    except:
        try: df = pd.read_csv(f, nrows=5, encoding='cp949')
        except: continue
    print(f"\n{'='*80}")
    print(f"{os.path.basename(f)} | 컬럼: {len(df.columns)}개")
    print(f"컬럼: {list(df.columns)}")
    display(df.head())


---
## 6. 전체 파일 요약


In [ ]:
all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        fp = os.path.join(root, f)
        all_files.append({
            '파일': os.path.relpath(fp, DATA_DIR),
            '용량(MB)': round(os.path.getsize(fp)/1024/1024, 1),
            '확장자': os.path.splitext(f)[1]
        })

df_all = pd.DataFrame(all_files).sort_values('용량(MB)', ascending=False)
print(f"전체 파일: {len(df_all)}개 | 총 용량: {df_all['용량(MB)'].sum()/1024:.1f}GB")
print(f"\n확장자별:")
print(df_all.groupby('확장자')['용량(MB)'].agg(['count','sum']).sort_values('sum', ascending=False))
print(f"\n상위 20개:")
display(df_all.head(20))
